# pandas library: data analysis and visualisation

In [ ]:
import pandas as pd

In [ ]:
movie: pd.DataFrame = pd.read_csv("data/movie.csv")

movie.head()

## Data analysis

### Ranking objects by an attribute

This operation selects $n$ rows having the largest or smallest value in a chosen column. Use the `nlargest` or `nsmallest` methods of a `DataFrame`, which return the $n$ rows with the largest or smallest values in the specified column, respectively.

Example: select the 10 films with the largest budgets:

In [ ]:
movie.nlargest(10, "budget")[["movie_title", "budget", "imdb_score", "genres", "title_year"]]

Similarly, the 10 films with the smallest budgets can be found as follows:

In [ ]:
movie.nsmallest(10, "budget")[["movie_title", "budget", "imdb_score", "genres", "title_year"]]

Combining the two methods allows objects to be ranked according to two attributes. For example, from the 10 least expensive films, we can select the 5 with the highest IMDb rating:

In [ ]:
movie.nsmallest(10, "budget")[["movie_title", "budget", "imdb_score", "genres", "title_year"]].nlargest(5, "imdb_score")

### Sorting data frames

Data frames, like database tables, can be sorted by one or more columns. Use the `sort_values` method of a `DataFrame`.

Example: sort films by duration in descending order.

In [ ]:
movie.sort_values("duration", ascending=False).head(15)

A data frame can also be sorted by several columns. Pass a collection of column names to `sort_values`. The following example sorts first by `duration` in descending order and then by `imdb_score`:

In [ ]:
movie.sort_values(["duration", "imdb_score"], ascending=False)[["movie_title", "duration", "imdb_score", "genres"]].head(20)

### Removing duplicates

Use `drop_duplicates` to remove duplicates, i.e. rows containing the same value in a selected column. Like the methods above, it does not overwrite the original data frame but returns a modified copy, so it can be used to filter or group the original data. For example, find the highest-rated film in each year:

In [ ]:
movie[["movie_title", "imdb_score", "title_year"]].sort_values(["title_year", "imdb_score"], ascending=False).drop_duplicates(subset="title_year")

### Data aggregation

Data aggregation groups rows by a selected column and then applies an aggregation function that computes values for selected attributes in each group. An aggregation function may, for example, compute a sum or arithmetic mean. Use `groupby` on a `DataFrame` to group rows; it returns a `DataFrameGroupBy` object. Then call `agg` on that grouped object to apply aggregation functions.

Example: mean film rating in each year.

In [ ]:
movie.groupby("title_year").agg({"imdb_score": "mean"}).sort_values("title_year", ascending=False)

The total budget of films made by each director can be obtained in the same way:

In [ ]:
movie.groupby("director_name").agg({"budget": "sum"}).sort_values("budget", ascending=False)

Similarly, for each country we can obtain the number of films produced and their mean and total budgets:

In [ ]:
movie.groupby("country").agg({"budget": ["sum", "mean"], "movie_title": "count"})

### Custom aggregation functions

pandas allows custom functions to be used during data aggregation. Such a function should accept one `Series` argument and return a scalar value.

Example: mean film score in each year using a custom aggregation function.

In [ ]:
def my_mean(x: pd.Series) -> float:
  return x.sum() / x.count()

In [ ]:
movie.groupby("title_year").agg({"imdb_score": my_mean}).sort_values("title_year", ascending=False)

### Filtering data

Use the `query` method of a `DataFrame` to filter data. Its argument is an expression of the form `[column_name] [logical_operator] [value]`, for example `country == "Poland"`, which selects all rows matching the condition.

Example: select all films directed by James Cameron.

In [ ]:
movie.query('director_name == "James Cameron"')

Logical expressions can be combined using `and` and `or` to create more complex filters.

Example: all James Cameron films produced before 1989.

In [ ]:
movie[["director_name", "movie_title", "title_year"]].query('director_name == "James Cameron" and title_year < 1989')

## Data visualisation

pandas provides a convenient interface for visualising data stored in data frames. Its plotting methods use `matplotlib` internally. To ensure plots are displayed correctly across environments, `matplotlib` should also be imported.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

### Line chart

The simplest line chart can be created by calling `plot` on a data series. Values from the row index are placed on the x-axis, while the series values are placed on the y-axis.

Example: line chart of mean film duration by year.

In [ ]:
ax: plt.Axes = movie.groupby("title_year").agg({"duration": "mean"}).plot()
ax.set_ylabel("srednia dlugosc")

plt.show()

pandas can also plot entire data frames, displaying several overlapping lines.

Example: mean film duration and number of films by year.

In [ ]:
ax: plt.Axes = movie.groupby("title_year").agg({"duration": "mean", "movie_title": "count"}).plot()

plt.show()

### Bar chart

Use the `bar` method to create bar charts in pandas.

Example: bar chart of mean film rating for each language.

In [ ]:
ax: plt.Axes = movie.groupby("language").agg({"imdb_score": "mean"}).plot.bar(figsize=(10, 5))
ax.set_ylabel("srednia ocena")

plt.show()

### Pie chart

Use the `pie` method to create a pie chart.

Example: pie chart showing the proportion of colour and black-and-white films among all films.

In [ ]:
ax: plt.Axes = movie.groupby("color").agg({"color": "count"}).plot.pie(subplots=True, autopct="%.2f %%", ylabel="", figsize=(5, 5))

plt.show()